# Audit du modèle sectoriel Europe

Ce notebook déroule le modèle étape par étape : dates, historique figé, sous-variables, piliers, macro, rate overlay, alignement, rangs, Top/Worst et recommandation finale.

Les calculs utilisent les mêmes fonctions que `model_secto_eu.py`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

import local_config as cfg
import model_secto_eu as modele

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')


## 1. Configuration utilisée


In [ ]:
print('Fichier sectoriel :', modele.FICHIER_EXCEL_PAR_DEFAUT)
print('Fichier macro     :', modele.FICHIER_MACRO_PAR_DEFAUT)
print('Nombre de secteurs:', len(cfg.SECTEURS))
print('Fenêtre historique:', cfg.FENETRE_HISTORIQUE, 'mois')
print('Top / Worst       :', cfg.N_TOP, '/', cfg.N_WORST)

display(pd.DataFrame(cfg.POIDS_REGIME).T)


In [ ]:
wb_eu = load_workbook(modele.FICHIER_EXCEL_PAR_DEFAUT, data_only=True, read_only=False, keep_vba=False)
wb_macro = load_workbook(modele.FICHIER_MACRO_PAR_DEFAUT, data_only=True, read_only=False, keep_vba=False)
print('Workbooks ouverts.')


## 2. Dates sources et dates modèle

Toute date est ramenée au dernier jour calendaire de son mois.


In [ ]:
lignes = []
for pilier, variables in cfg.VARIABLES_HISTORIQUES.items():
    premiere_variable = next(iter(variables.values()))
    ws = wb_eu[premiere_variable['sheet']]
    for ligne in range(8, 13):
        date_source = ws.cell(ligne, 1).value
        lignes.append({
            'pilier': pilier,
            'sheet': ws.title,
            'ligne': ligne,
            'date_source': date_source,
            'date_modele': modele.normaliser_date_mensuelle(date_source),
        })
display(pd.DataFrame(lignes))


## 3. Historique figé

`date` est la clé mensuelle. `date_source` conserve la date observée lors de la première insertion.


In [ ]:
dossier_historique = Path(modele.__file__).resolve().parent / cfg.CONFIG_HISTORIQUE['dossier']
print('Nombre de séries enregistrées :', len(list(dossier_historique.glob('*.csv'))))
exemple = dossier_historique / 'Leverage_FMA_AF.csv'
if exemple.exists():
    display(pd.read_csv(exemple).head(10))


## 4. Exemple détaillé : Net Debt / EBITDA

Chaîne : ratio sectoriel → écart à la moyenne → rang historique → score 0-10.


In [ ]:
config_variable = cfg.VARIABLES_HISTORIQUES['Leverage']['net_debt_ebitda']
raw = modele.lire_dates_et_bloc(wb_eu[config_variable['sheet']], config_variable['colonne'])
diff = modele.calculer_diff_vs_moyenne(raw, config_variable['moyenne_sans_finance'])
score = modele.calculer_score_historique(diff, config_variable['ordre_rank'], config_variable.get('fenetre_par_secteur'))
date_exemple = raw.index.max()
display(pd.DataFrame({
    'ratio_brut': raw.loc[date_exemple],
    'diff_vs_moyenne': diff.loc[date_exemple],
    'score_0_10': score.loc[date_exemple],
}))


## 5. Leverage / Margin / Value / Growth


In [ ]:
piliers_historiques, sous_scores = modele.calculer_piliers_historiques(wb_eu)
for nom, df in piliers_historiques.items():
    date = df.index.max()
    print('\n', nom, date.date())
    display(df.loc[[date]])


## 6. Momentum


In [ ]:
momentum, sous_momentum = modele.calculer_momentum(wb_eu)
date_momentum = momentum.index.max()
display(pd.DataFrame({
    '6m_1m': sous_momentum['momentum_6m_1m'].loc[date_momentum],
    '12m_1m': sous_momentum['momentum_12m_1m'].loc[date_momentum],
    'revision': sous_momentum['earnings_revision_ratio'].loc[date_momentum],
    'momentum_final': momentum.loc[date_momentum],
}))


## 7. Volatility


In [ ]:
volatility, sous_vol, retours = modele.calculer_volatilite(wb_eu)
date_vol = volatility.index.max()
display(pd.DataFrame({
    'volatility_6m': sous_vol['volatility_6m'].loc[date_vol],
    'downside_volatility_18m': sous_vol['downside_volatility_18m'].loc[date_vol],
    'volatility_final': volatility.loc[date_vol],
}))


## 8. Alignement des six piliers


In [ ]:
piliers = dict(piliers_historiques)
piliers['Momentum'] = momentum
piliers['Volatility'] = volatility

resume_dates = pd.DataFrame([
    {'pilier': nom, 'date_min': df.index.min(), 'date_max': df.index.max(), 'nombre_mois': len(df.index.unique())}
    for nom, df in piliers.items()
])
display(resume_dates)

piliers_alignes = modele.aligner_piliers(piliers)
dates_communes = next(iter(piliers_alignes.values())).index
print('Dernières dates communes :', [x.strftime('%Y-%m-%d') for x in dates_communes[:12]])
date_piliers = dates_communes.max()
display(pd.DataFrame({nom: df.loc[date_piliers] for nom, df in piliers_alignes.items()}))


## 9. Rangs des piliers


In [ ]:
rangs = modele.calculer_rangs_piliers(piliers_alignes)
display(pd.DataFrame({nom: df.loc[date_piliers] for nom, df in rangs.items()}))


## 10. Macro externe


In [ ]:
macro_externe = modele.lire_macro_externe(wb_macro)
display(macro_externe.tail(12))


## 11. Rate overlay


In [ ]:
taux_us10y = modele.lire_taux_us10y(wb_eu)
signal_taux = modele.calculer_signal_taux(taux_us10y)
display(signal_taux.tail(12))


## 12. Contexte macro complet


In [ ]:
contexte_macro = modele.construire_contexte_macro(wb_eu, wb_macro)
display(contexte_macro.tail(12))


## 13. Dernière date complète de recommandation


In [ ]:
dates_macro_valides = contexte_macro[
    contexte_macro['cycle'].isin(cfg.POIDS_REGIME)
    & contexte_macro['signal_taux'].isin(['On','Off'])
].index
dates_reco = dates_communes.intersection(dates_macro_valides).sort_values()
if len(dates_reco) == 0:
    raise ValueError('Aucune date complète pour la recommandation.')
date_reco = dates_reco[-1]
print('Date auditée :', date_reco.date())
display(contexte_macro.loc[[date_reco]])


## 14. Score de base et tilt de régime


In [ ]:
regime = contexte_macro.at[date_reco, 'cycle']
score_base = modele.calculer_score_pondere(rangs, cfg.POIDS_BASE, date_reco)
rang_base = modele.rang_secteurs(score_base)
score_tilt = modele.calculer_score_pondere(rangs, cfg.POIDS_REGIME[regime], date_reco)
rang_tilt = modele.rang_secteurs(score_tilt)
display(pd.DataFrame({
    'score_base': score_base,
    'rang_base': rang_base,
    'score_tilt': score_tilt,
    'rang_tilt': rang_tilt,
}).sort_values('rang_tilt', ascending=False))


## 15. Score global


In [ ]:
n_composantes = len(cfg.POIDS_BASE) + 1
score_global = pd.Series(0.0, index=cfg.SECTEURS)
for pilier in cfg.POIDS_BASE:
    score_global += rangs[pilier].loc[date_reco] / n_composantes
score_global += rang_tilt / n_composantes
rang_global = modele.rang_secteurs(score_global)
display(pd.DataFrame({'score_global': score_global, 'rang_global': rang_global}).sort_values('rang_global', ascending=False))


## 16. Comptage Top / Worst


In [ ]:
top_count = pd.Series(0, index=cfg.SECTEURS, dtype=int)
bottom_count = pd.Series(0, index=cfg.SECTEURS, dtype=int)
for pilier in cfg.POIDS_BASE:
    r = rangs[pilier].loc[date_reco]
    top_count += (r > len(cfg.SECTEURS) - cfg.N_TOP).fillna(False).astype(int)
    bottom_count += (r <= cfg.N_WORST).fillna(False).astype(int)
top_count += cfg.POIDS_VOTE_MACRO * (rang_tilt > len(cfg.SECTEURS) - cfg.N_TOP).fillna(False).astype(int)
bottom_count += cfg.POIDS_VOTE_MACRO * (rang_tilt <= cfg.N_WORST).fillna(False).astype(int)
if contexte_macro.at[date_reco, 'signal_taux'] == 'On':
    for pilier in cfg.PILIERS_RATE_OVERLAY:
        r = rangs[pilier].loc[date_reco]
        top_count += (r > len(cfg.SECTEURS) - cfg.N_TOP).fillna(False).astype(int)
        bottom_count += (r <= cfg.N_WORST).fillna(False).astype(int)
for secteur in cfg.SECTEURS_SANS_VOTE_TOP_WORST:
    top_count[secteur] = 0
    bottom_count[secteur] = 0
display(pd.DataFrame({'top_count': top_count, 'bottom_count': bottom_count, 'rang_global': rang_global}).sort_values(['top_count','rang_global'], ascending=[False,False]))


## 17. Recommandation finale


In [ ]:
top, worst = modele.choisir_top_worst(top_count, bottom_count, rang_global)
recommandation = pd.Series('Neutral', index=cfg.SECTEURS)
recommandation.loc[top] = 'Positive'
recommandation.loc[worst] = 'Negative'
print('Top :', top)
print('Worst :', worst)
display(pd.DataFrame({
    'rang_global': rang_global,
    'top_count': top_count,
    'bottom_count': bottom_count,
    'recommendation': recommandation,
}).sort_values('rang_global', ascending=False))


## 18. Contrôle avec calculer_modele()


In [ ]:
resultats = modele.calculer_modele()
date_finale = resultats['historique']['date'].max()
final = resultats['historique'][resultats['historique']['date'] == date_finale].sort_values('rang_global', ascending=False)
display(final[['date','secteur','macro_score','cycle','signal_taux','rang_global','top_count','bottom_count','recommendation']])


In [ ]:
wb_eu.close()
wb_macro.close()
print('Audit terminé.')
